In [ ]:
!pip install dspy-ai

In [ ]:
import dspy
import random
from dspy import LM
import pandas as pd
from datasets import load_dataset
from getpass import getpass
import os

# Enter your Groq API key securely
os.environ["GROQ_API_KEY"] = getpass("Enter GROQ API Key: ")
groq_lm = LM(model="groq/llama-3.1-8b-instant")



# Configure DSPy
dspy.settings.configure(
    lm=groq_lm,

)



In [ ]:
pip install datasets pandas scikit-learn

In [ ]:
import pandas as pd
from datasets import load_dataset

# Load dataset (correct version)
dataset = load_dataset("clinc_oos", "plus")
intent_names = dataset["train"].features["intent"].names

print(dataset)

In [ ]:
df_train = dataset["train"].to_pandas()
df_test = dataset["test"].to_pandas()

small_test = df_test.head(100)

print(df_train.head())

In [ ]:
# Convert numeric labels into readable intent names
df_train["intent"] = df_train["intent"].apply(lambda x: intent_names[x])
df_test["intent"] = df_test["intent"].apply(lambda x: intent_names[x])

In [ ]:

df_train.iloc[0]

In [ ]:
## run prediction
first_row = df_train.iloc[0]
print(f"customer message: {first_row['text']},real class: {first_row['intent']}")


In [ ]:
import random
import dspy

# Get unique labels from dataframe
labels = df_train["intent"].unique().tolist()
labels_str = ", ".join(labels)

def get_dspy_examples(df, k):
    dspy_examples = []

    labels = df["intent"].unique().tolist()

    for label in labels:
        label_df = df[df["intent"] == label].sample(n=k, random_state=42)

        for _, row in label_df.iterrows():
            dspy_examples.append(
                dspy.Example(
                    customer_message=row["text"],
                    answer=row["intent"]
                ).with_inputs("customer_message")
            )

    return dspy_examples
train_examples = get_dspy_examples(df_train, k=2)
all_test_examples = get_dspy_examples(df_test, k=10)

dev_examples = random.sample(all_test_examples, len(all_test_examples) // 2)
test_examples = [e for e in all_test_examples if e not in dev_examples]

In [ ]:
import dspy

class IntentSignature(dspy.Signature):
    """
    You are an intent classification system.

    Possible labels:
    translate
    transfer
    balance
    greeting
    complaint
    goodbye

    Output ONLY one label from the list above.
    Do NOT explain.
    Do NOT answer the question.
    """

    customer_message = dspy.InputField()
    answer = dspy.OutputField(desc="One label only.")

In [ ]:
# Baseline predictor
baseline_program = dspy.Predict(IntentSignature)

# Chain-of-thought predictor (needed for Bootstrap)
cot_program = dspy.ChainOfThought(IntentSignature)

In [ ]:
import random

def get_dspy_examples(df, k):
    dspy_examples = []

    labels = df["intent"].unique().tolist()

    for label in labels:
        label_df = df[df["intent"] == label].sample(n=k, random_state=42)

        for _, row in label_df.iterrows():
            dspy_examples.append(
                dspy.Example(
                    customer_message=row["text"],
                    answer=row["intent"]
                ).with_inputs("customer_message")
            )

    return dspy_examples


# Create train and test examples
train_examples = get_dspy_examples(df_train, k=2)
test_examples = get_dspy_examples(df_test, k=2)

In [ ]:
print("Baseline Predictions:\n")

for ex in test_examples[:3]:
    pred = baseline_program(customer_message=ex.customer_message)
    print("Message:", ex.customer_message)
    print("True:", ex.answer)
    print("Predicted:", pred.answer)
    print("------")

In [ ]:
from dspy.teleprompt import LabeledFewShot

labeled_optimizer = LabeledFewShot(k=5)

labeled_model = labeled_optimizer.compile(
    baseline_program,
    trainset=train_examples
)

In [ ]:
def metric(example, prediction, trace=None):
    return prediction.answer.strip().lower() == example.answer.strip().lower()

In [ ]:
print("Testing LabeledFewShot:\n")

for ex in test_examples[:3]:
    pred = labeled_model(**ex.inputs())
    print("Message:", ex.customer_message)
    print("True:", ex.answer)
    print("Predicted:", pred.answer)
    print("------")

In [ ]:
from dspy.teleprompt import BootstrapFewShot

bootstrap_optimizer = BootstrapFewShot(metric=metric)

optimized_model = bootstrap_optimizer.compile(
    cot_program,
    trainset=train_examples
)

In [ ]:
import time

def evaluate_model(model, examples, n=5):
    correct = 0

    for ex in examples[:n]:
        pred = model(customer_message=ex.customer_message)

        predicted = pred.answer.strip().lower()
        true = ex.answer.strip().lower()

        print("Message:", ex.customer_message)
        print("True:", true)
        print("Predicted:", predicted)
        print("------")

        if predicted == true:
            correct += 1

        time.sleep(2)  # prevent rate limit

    return correct / n

In [ ]:
for ex in test_examples[:3]:
    pred = baseline_program(customer_message=ex.customer_message)
    print("TRUE:", ex.answer)
    print("PRED:", pred.answer)
    print("------")

In [ ]:
baseline_acc = evaluate_model(baseline_program, test_examples, n=3)
labeled_acc = evaluate_model(labeled_model, test_examples, n=3)
bootstrap_acc = evaluate_model(optimized_model, test_examples, n=3)

print("\nFINAL COMPARISON")
print("Baseline:", baseline_acc)
print("Labeled FewShot:", labeled_acc)
print("Bootstrap FewShot:", bootstrap_acc)

In [ ]:
user_text = input("Enter customer message: ")

prediction = optimized_model(customer_message=user_text)

print("Predicted Intent:", prediction.answer)